In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

In [6]:
NUM_INPUT = 784
NUM_OUTPUT = 50
DT = 0.005
T_MAX = 0.350

def assign_labels():
    print("Loading weights and training subset via sklearn...")
    weights = np.load('vdsp_weights.npy')
    
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')
    
    # Use the first 5000 images from the training set for label assignment
    images = mnist.data[:5000] / 255.0
    labels = mnist.target[:5000].astype(int)
    
    spike_counts = np.zeros((NUM_OUTPUT, 10))
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    
    for img_idx, img in enumerate(images):
        v_pre.fill(0.0)
        v_post.fill(0.0)
        img_label = labels[img_idx]
        
        for t in np.arange(0, T_MAX, DT):
            v_pre += (DT / 0.03) * (-v_pre + img + 0.5)
            spikes_pre = v_pre >= 1.0
            v_pre[spikes_pre] = -1.0
            
            v_post += (DT / 0.03) * (-v_post + (spikes_pre.astype(float) @ weights))
            spikes_post = v_post >= 1.0
            
            if spikes_post.any():
                winner = np.argmax(v_post)
                spike_counts[winner, img_label] += 1
                v_post[:] = 0.0 
                
    assigned_labels = np.argmax(spike_counts, axis=1)
    np.save('assigned_labels.npy', assigned_labels)
    print(f"Assigned Labels: {assigned_labels}")

In [7]:
assign_labels()

Loading weights and training subset via sklearn...
Assigned Labels: [7 5 0 0 3 8 0 5 9 8 6 2 0 2 8 2 3 4 7 3 0 3 0 4 2 3 5 9 0 1 6 6 5 6 8 7 4
 2 5 4 0 3 2 0 7 4 2 2 9 4]
